# Concatenate all batches per model into one single file

In [ ]:
import os
import pandas as pd
import glob

# Handle both script and notebook execution
try:
    # For script execution
    project_root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    # For notebook execution or other environments where __file__ is not defined
    # Use current working directory instead
    print("Running in a notebook or environment where __file__ is not defined")
    print("Using current working directory as reference")
    
    # Get the current working directory
    current_dir = os.getcwd()
    
    # If we're already in the Analysis folder
    if os.path.basename(current_dir) == "Analysis":
        project_root = os.path.dirname(current_dir)
    else:
        # Assume we're at the project root
        project_root = current_dir
    
    print(f"Project root: {project_root}")

# Define paths
raw_data_path = os.path.join(project_root, "Data", "Batches")
results_path = os.path.join(project_root, "Data", "Results")
os.makedirs(results_path, exist_ok=True)

# Check if directories exist
if not os.path.exists(raw_data_path):
    raise FileNotFoundError(f"Batches directory not found: {raw_data_path}")
    
print(f"Processing data from: {raw_data_path}")
print(f"Results will be saved to: {results_path}")

# Process each subfolder in raw data
for folder in os.listdir(raw_data_path):
    folder_path = os.path.join(raw_data_path, folder)
    
    # Skip if not a directory
    if not os.path.isdir(folder_path):
        continue
    
    # Get all CSV files in the folder
    csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
    
    if not csv_files:
        continue
    
    # Concatenate all CSVs in the folder
    dfs = []
    for file in csv_files:
        df = pd.read_csv(file)
        dfs.append(df)
    
    if dfs:
        # Combine all dataframes
        combined_df = pd.concat(dfs, ignore_index=True)
        
        # Save to results folder with folder name
        output_file = os.path.join(results_path, f"{folder}.csv")
        combined_df.to_csv(output_file, index=False)
        
print("Data cleaning complete")

# Check for missing data

In [ ]:
import os
import pandas as pd
import glob
from tabulate import tabulate

# Handle path detection for notebooks and scripts
try: project_root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError: project_root = os.getcwd() if os.path.basename(os.getcwd()) != "Analysis" else os.path.dirname(os.getcwd())

# Setup
results_path = os.path.join(project_root, "Data", "Results")
csv_files = glob.glob(os.path.join(results_path, "*.csv"))
missing_vars = ['answer.Q1_S_Before_list', 'answer.Q2_L_Before_list', 'answer.Q2_L_After', 'answer.Q1_S_After']

# Process each CSV file
for csv_file in csv_files:
    filename = os.path.basename(csv_file)
    print(f"\n===== Processing {filename} =====")
    
    df = pd.read_csv(csv_file)
    print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
    
    # Check for duplicates
    if 'agent.userid' in df.columns:
        total = df['agent.userid'].count()
        unique = df['agent.userid'].nunique()
        dupes = df[df.duplicated('agent.userid', keep=False)]['agent.userid'].unique()
        print(f"User IDs: {total} total, {unique} unique, {len(dupes)} duplicated")
        
    # Treatment frequency table
    if 'scenario.treatment' in df.columns:
        t_counts = df['scenario.treatment'].value_counts(dropna=False)
        t_pcts = df['scenario.treatment'].value_counts(normalize=True, dropna=False) * 100
        t_table = pd.DataFrame({'Count': t_counts, 'Percentage': t_pcts}).reset_index().rename(columns={'index': 'scenario.treatment'})
        
        # Add missing counts by treatment
        for var in missing_vars:
            if var in df.columns:
                missing_by_t = df.groupby('scenario.treatment')[var].apply(lambda x: x.isna().sum())
                t_table[f'Missing {var}'] = t_table['scenario.treatment'].map(missing_by_t).fillna(0).astype(int)
        
        print("\nTreatment frequency table:")
        print(tabulate(t_table, headers='keys', tablefmt='grid', showindex=False))
    
    # Missing values summary
    print("\nMissing values summary:")
    missing_summary = [[var, df.shape[0], df[var].isna().sum(), f"{df[var].isna().sum()/df.shape[0]*100:.2f}%"] 
                       if var in df.columns else [var, "-", "-", "Not found"] for var in missing_vars]
    print(tabulate(missing_summary, headers=['Variable', 'Total', 'Missing', '%'], tablefmt='grid'))